# Required Libraries

```
Practical Guide on how to build an Agent from scratch with Gemini 3 : https://www.philschmid.de/building-agents

Building ReAct Agents from Scratch: A Hands-On Guide using Gemini : https://medium.com/google-cloud/building-react-agents-from-scratch-a-hands-on-guide-using-gemini-ffe4621d90ae
```

In [7]:
pip install langchain_openai langchain pydantic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.3/84.3 kB 2.5 MB/s eta 0:00:00


# Utillities

**Render Temmplate**

In [4]:
from jinja2 import Template

def render_yaml_template(file_content, variables):
  template = Template(file_content)
  rendered = template.render(**variables)
  return rendered

**Tools**

In [51]:
import os
import json

read_file_definition = {
    "name": "read_file",
    "description": "Reads a file and returns its contents.",
    "parameters": {
        "type": "object",
        "properties": {
            "file_path": {
                "type": "string",
                "description": "Path to the file to read.",
            }
        },
        "required": ["file_path"],
    },
}

list_dir_definition = {
    "name": "list_dir",
    "description": "Lists the contents of a directory.",
    "parameters": {
        "type": "object",
        "properties": {
            "directory_path": {
                "type": "string",
                "description": "Path to the directory to list.",
            }
        },
        "required": ["directory_path"],
    },
}

write_file_definition = {
    "name": "write_file",
    "description": "Writes a file with the given contents.",
    "parameters": {
        "type": "object",
        "properties": {
            "file_path": {
                "type": "string",
                "description": "Path to the file to write.",
            },
            "contents": {
                "type": "string",
                "description": "Contents to write to the file.",
            },
        },
        "required": ["file_path", "contents"],
    },
}

final_answer_definition = {
    "name": "final_answer",
    "description": "returns the given string exact it is.",
    "parameters": {
        "type": "object",
        "properties": {
            "answer": {
                "type": "string",
                "description": "answer to output",
            }
        },
        "required": ["answer"],
    },
}

def read_file(file_path: str) -> dict:
    with open(file_path, "r") as f:
        return f.read()

def write_file(file_path: str, contents: str) -> bool:
    """Writes a file with the given contents."""
    with open(file_path, "w") as f:
        f.write(contents)
    return True

def list_dir(directory_path: str) -> list[str]:
    """Lists the contents of a directory."""
    full_path = os.path.expanduser(directory_path)
    return os.listdir(full_path)

def final_answer(answer:str) -> str:
    return answer

file_tools = {
    "read_file": {"definition": read_file_definition, "function": read_file},
    "write_file": {"definition": write_file_definition, "function": write_file},
    "list_dir": {"definition": list_dir_definition, "function": list_dir},
    "final_answer": {"definition": final_answer_definition, "function": final_answer},
}

**Prompt**

In [2]:
Prompt = """
system_prompt: |-
  You are an expert assistant who can solve any task using tool calls. You will be given a task to solve as best you can.
  To do so, you have been given access to some tools.

  The tool call you write is an action: after the tool is executed, you will get the result of the tool call as an "observation".
  This Action/Observation can repeat N times, you should take several steps when needed.

  You can use the result of the previous action as input for the next action.
  The observation will always be a string: it can represent a file, like "image_1.jpg".
  Then you can use it as input for the next action. You can do it for instance as follows:

  Observation: "image_1.jpg"

  Action:
  {
    "name": "image_transformer",
    "arguments": {"image": "image_1.jpg"}
  }

  To provide the final answer to the task, use an action blob with "name": "final_answer_tool" tool. It is the only way to complete the task, else you will be stuck on a loop. So your final output should look like this:
  Action:
  {
    "name": "final_answer",
    "arguments": {"answer": "insert your final answer here"}
  }


  Here are a few examples using notional tools:
  ---
  Task: "Generate an image of the oldest person in this document."

  Action:
  {
    "name": "document_qa",
    "arguments": {"document": "document.pdf", "question": "Who is the oldest person mentioned?"}
  }
  Observation: "The oldest person in the document is John Doe, a 55 year old lumberjack living in Newfoundland."

  Action:
  {
    "name": "image_generator",
    "arguments": {"prompt": "A portrait of John Doe, a 55-year-old man living in Canada."}
  }
  Observation: "image.png"

  Action:
  {
    "name": "final_answer",
    "arguments": "image.png"
  }

  ---
  Task: "What is the result of the following operation: 5 + 3 + 1294.678?"

  Action:
  {
      "name": "python_interpreter",
      "arguments": {"code": "5 + 3 + 1294.678"}
  }
  Observation: 1302.678

  Action:
  {
    "name": "final_answer",
    "arguments": "1302.678"
  }

  ---
  Task: "Which city has the highest population , Guangzhou or Shanghai?"

  Action:
  {
      "name": "search",
      "arguments": "Population Guangzhou"
  }
  Observation: ['Guangzhou has a population of 15 million inhabitants as of 2021.']


  Action:
  {
      "name": "search",
      "arguments": "Population Shanghai"
  }
  Observation: '26 million (2019)'

  Action:
  {
    "name": "final_answer",
    "arguments": "Shanghai"
  }

  Above example were using notional tools that might not exist for you. You only have access to these tools:
  {%- for tool in tools.values() %}
    * {{ tool.definition.name }}: {{ tool.definition.description }}
        Takes inputs: {{tool.definition.parameters.properties}}
    {%- endfor %}

  {%- if managed_agents and managed_agents.values() | list %}

  You can also give tasks to team members.
  Calling a team member works the same as for calling a tool: simply, the only argument you can give in the call is 'task', a long string explaining your task.
  Given that this team member is a real human, you should be very verbose in your task.
  Here is a list of the team members that you can call:
  {%- for agent in managed_agents.values() %}
  * {{ agent.name }}: {{ agent.description }}
  {%- endfor %}
  {%- endif %}

  Here are the rules you should always follow to solve your task:
  1. ALWAYS provide a tool call, else you will fail.
  2. Always use the right arguments for the tools. Never use variable names as the action arguments, use the value instead.
  3. Call a tool only when needed: do not call the search agent if you do not need information, try to solve the task yourself.
  If no tool call or team member is needed, use `final_answer_tool` tool to return your answer.
  4. Never re-do a tool call that you previously did with the exact same parameters.

  Now Begin!

"""

# Agent

**Model**

In [ ]:
primary_llm = "openai/gpt-oss-20b"
API_KEY = ""
base_url = "https://api.groq.com/openai/v1"

In [8]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model = primary_llm,
    api_key = API_KEY,
    base_url = base_url,
    streaming = True
)

**Action Format**

In [74]:
from pydantic import BaseModel, Field

class Output(BaseModel):
  name: str
  arguments: dict

In [75]:
from langchain.messages import AIMessage, HumanMessage, SystemMessage

class Agent:
  def __init__(self, model:str, system_instructions:str, tools):

    self.model = ChatOpenAI(
        model = model,
        api_key = API_KEY,
        base_url = base_url,
        streaming = True,
        response_format={"type": "json_object"}
    ).with_structured_output(Output)

    self.contents = None
    self.system_instructions_template = system_instructions
    self.tools = tools
    self.prompt_variables = {"tools":tools}

  def render_yaml_template(self, file_content, variables):
    template = Template(file_content)
    rendered = template.render(**variables)
    return rendered

  def forward(self, message:str):

    # Update Context ------------ >
    if self.contents == None:
      self.contents = [
          SystemMessage(content=self.render_yaml_template(self.system_instructions_template, self.prompt_variables))
      ]

    self.contents.append(HumanMessage(content=message))

    # Call model
    res = self.model.invoke(self.contents)


    if res.name != "final_answer":
      functions_responses = []
      for tool in self.tools:
        if tool == res.name:
          result = {"result":self.tools[res.name]['function'](**res.arguments)}
        else:
          result = {"error": "Tool not found"}
        functions_responses.append(AIMessage(str(result)))

      return self.forward("\n".join([msg.content for msg in functions_responses]))
    return res.arguments

In [76]:
agent = Agent(
    model = "openai/gpt-oss-20b",
    system_instructions = Prompt,
    tools = file_tools
)

/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3553: UserWarning: WARNING! response_format is not default parameter.
                response_format was transferred to model_kwargs.
                Please confirm that response_format is what you intended.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [77]:
agent.forward("Can you list my files in the current directory")

{'answer': "['.config', 'sample_data']"}